# Connecting Flet to FastAPI

Parts II and III of this series built the backend: photo storage in S3, metadata and vector indexes in Postgres, a FastAPI service exposing REST endpoints for upload, search, and people clustering. In Part IV we build the Flet desktop frontend that talks to this backend.

Frontend-backend integration is not just a matter of calling `httpx.get`. It involves a coherent auth story (how does the Flet app prove it is acting on behalf of a logged-in user?), a mental model for loading remote assets efficiently (presigned URLs rather than proxying bytes through the API), and UI patterns that make the latency of network calls invisible to the user: optimistic updates and lazy loading. This notebook covers all three.

:::{.callout-important}
Flet UI code that involves rendering cannot execute inside a Jupyter notebook. All Flet component code in this notebook is presented as static `{.python filename="..."}` code blocks. Runnable cells are used exclusively for `httpx` client calls, JWT logic, Pydantic models, and other non-render Python logic.

:::

## The Integration Architecture

The Flet desktop app runs as a local Python process. The FastAPI service runs on `localhost:8000` in development (and on a remote host in production). Communication between the two happens over HTTP using `httpx.AsyncClient`.

Three design decisions shape the integration:

**Auth: JWT access tokens.** The Flet login screen calls `POST /auth/token` with username and password and receives a short-lived JWT. The token is stored in `page.shared_preferences` (Flet's local key-value store) and included as `Authorization: Bearer <token>` on every subsequent request. When the token expires the app redirects to the login screen.

**Asset delivery: presigned S3 URLs.** Photos are not proxied through the FastAPI service. Instead, API responses include presigned S3 URLs — HTTPS links that grant temporary read access to a specific S3 object, signed with the server's AWS credentials. The Flet `Image` control loads the URL directly from S3, bypassing the API entirely. This keeps the FastAPI service stateless and avoids a bandwidth bottleneck.

**Real-time: WebSocket.** Upload progress, clustering job status, and new-photo notifications are pushed from the server to the client over a WebSocket connection. This is covered in notebook 12; for now we focus on the REST layer.

## JWT Auth Flow

A **JWT** (JSON Web Token) is a compact, self-contained token that encodes a set of *claims* (assertions about the authenticated entity) in a structure that the server can verify without a database lookup. Its format is `header.payload.signature`, each part base64url-encoded.

The **payload** carries standard claims:
- `sub` (subject): user ID or username.
- `exp` (expiration): Unix timestamp after which the token is invalid.
- `iat` (issued at): Unix timestamp of token creation.

The **signature** is `HMACSHA256(header + "." + payload, secret_key)`. Any server holding the same `secret_key` can verify the token without talking to a database, making JWT stateless.

The auth flow proceeds in four steps: (1) Flet sends credentials to `POST /auth/token`, (2) FastAPI verifies the password hash and returns a JWT, (3) Flet stores the JWT in `page.shared_preferences`, (4) every subsequent API call includes `Authorization: Bearer <token>` and the FastAPI dependency `get_current_user` decodes the JWT to identify the caller.

Implementing `create_access_token` and `decode_token`:

In [ ]:
from datetime import datetime, timedelta, timezone
from jose import jwt, JWTError

SECRET_KEY = "dev-secret-do-not-use-in-production"
ALGORITHM  = "HS256"
DEFAULT_EXPIRY_MINUTES = 60


def create_access_token(
    sub: str,
    expires_delta: timedelta = timedelta(minutes=DEFAULT_EXPIRY_MINUTES),
) -> str:
    """Create a signed JWT with `sub` claim and an expiry."""
    now = datetime.now(timezone.utc)
    payload = {
        "sub": sub,
        "iat": now,
        "exp": now + expires_delta,
    }
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)


def decode_token(token: str) -> dict:
    """Decode and verify a JWT. Raises JWTError on invalid/expired tokens."""
    return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])

Verifying the encode/decode round-trip:

In [ ]:
token  = create_access_token(sub="user-42")
claims = decode_token(token)

print(f"Token   : {token[:40]}...")
print(f"Subject : {claims['sub']}")
print(f"Expires : {datetime.fromtimestamp(claims['exp'], tz=timezone.utc).isoformat()}")

The corresponding FastAPI dependency that every protected endpoint uses:

In [ ]:
FASTAPI_AUTH = '''
from fastapi import Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer
from jose import JWTError

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/auth/token")


async def get_current_user(token: str = Depends(oauth2_scheme)) -> str:
    """FastAPI dependency: decode the Bearer token and return the user ID."""
    try:
        claims = decode_token(token)
        return claims["sub"]
    except (JWTError, KeyError):
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid or expired token",
            headers={"WWW-Authenticate": "Bearer"},
        )
'''
print(FASTAPI_AUTH.strip())

:::{.callout-caution}
Store `SECRET_KEY` in an environment variable, never in source code. A leaked secret key allows anyone to forge valid tokens for any `sub`.

:::

## The API Client

Rather than scattering `httpx` calls throughout the Flet component tree, they are centralized in an `APIClient` class. The client does three things automatically: (1) it injects the `Authorization: Bearer {token}` header on every request so no individual call site has to handle auth, (2) it raises a typed `AuthError` sentinel on `401 Unauthorized` that the app-level error handler catches and translates into a redirect to the login screen, and (3) it retries transient `5xx` errors with exponential backoff using `tenacity`.

**Why a class, not a module-level function.** The `APIClient` holds the `base_url` and `token` as instance state, which lets it be injected into components as a constructor argument and easily replaced with a mock in tests. A module-level function with default parameters would require either global state or threading.local tricks to achieve the same goal.

**Retry policy.** `tenacity.retry` with `wait_exponential(min=1, max=30)` retries up to three times with 1s, 2s, and 4s delays. Server errors (`5xx`) are retried; client errors (`4xx`) are not — a `400 Bad Request` will not succeed on retry. Network errors (`httpx.ConnectError`, `httpx.TimeoutException`) are retried because they typically indicate transient infrastructure issues rather than a bug in the request.

Defining `APIClient` with auth injection, 401 handling, and retry:

In [ ]:
import httpx
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception


class AuthError(Exception):
    """Raised when the server returns 401; triggers redirect to login."""


def _is_server_error(exc: BaseException) -> bool:
    return isinstance(exc, httpx.HTTPStatusError) and exc.response.status_code >= 500


class APIClient:
    """Async HTTP client wrapping httpx; injects auth and handles errors."""

    def __init__(self, base_url: str, token: str):
        self._base_url = base_url
        self._token    = token

    def _headers(self) -> dict:
        return {"Authorization": f"Bearer {self._token}"}

    def _check(self, response: httpx.Response) -> httpx.Response:
        if response.status_code == 401:
            raise AuthError("Token expired or invalid")
        response.raise_for_status()
        return response

    @retry(
        retry=retry_if_exception(_is_server_error),
        wait=wait_exponential(multiplier=0.5, min=0.5, max=8),
        stop=stop_after_attempt(3),
        reraise=True,
    )
    async def get(self, path: str, **kwargs) -> dict:
        async with httpx.AsyncClient(base_url=self._base_url) as client:
            r = await client.get(path, headers=self._headers(), **kwargs)
        return self._check(r).json()

    @retry(
        retry=retry_if_exception(_is_server_error),
        wait=wait_exponential(multiplier=0.5, min=0.5, max=8),
        stop=stop_after_attempt(3),
        reraise=True,
    )
    async def post(self, path: str, json: dict | None = None, **kwargs) -> dict:
        async with httpx.AsyncClient(base_url=self._base_url) as client:
            r = await client.post(path, headers=self._headers(), json=json, **kwargs)
        return self._check(r).json()

    async def patch(self, path: str, json: dict | None = None, **kwargs) -> dict:
        async with httpx.AsyncClient(base_url=self._base_url) as client:
            r = await client.patch(path, headers=self._headers(), json=json, **kwargs)
        return self._check(r).json()

    async def delete(self, path: str, **kwargs) -> None:
        async with httpx.AsyncClient(base_url=self._base_url) as client:
            r = await client.delete(path, headers=self._headers(), **kwargs)
        self._check(r)

Exercising the client against the local FastAPI dev server (requires the server to be running):

In [ ]:
import asyncio

async def demo_client():
    token  = create_access_token(sub="demo-user")
    client = APIClient(base_url="http://localhost:8000", token=token)
    try:
        result = await client.get("/photos/?limit=3")
        print(f"Got {len(result)} photos")
    except httpx.ConnectError:
        print("FastAPI server not running — skipping live call")
    except AuthError:
        print("Auth error (token rejected)")

await demo_client()

## Paginated Gallery View

The gallery is the app's home screen: a `GridView` of photo thumbnails that loads from the API in pages of 50 and extends as the user scrolls. Each thumbnail is an `Image` control whose `src` is a presigned S3 URL returned by the API.

**Infinite scroll.** We attach an `on_scroll` handler to the `GridView`. When `e.pixels >= e.max_scroll_extent * 0.8` (meaning the user has scrolled to 80% of the current content), we fetch the next page and append new tiles. This is an eager prefetch strategy that hides the load latency.

**Thumbnail loading.** Each photo cell shows a `ProgressRing` until the `Image` control fires `on_load`, at which point the ring is replaced by the image. This is managed with `visible` toggling on a `Stack`: the ring and the image are both in the stack; only one is visible at a time.

```{.python filename="src/gallery.py"}
import flet as ft
import asyncio
from dataclasses import dataclass, field


@dataclass
class GalleryState:
    photos : list[dict] = field(default_factory=list)
    offset : int = 0
    loading: bool = False
    done   : bool = False  # no more pages

LIMIT = 50


@ft.component
def PhotoTile(photo: dict, on_click):
    loaded, set_loaded = ft.use_state(False)

    ring = ft.ProgressRing(width=32, height=32, stroke_width=3, visible=not loaded)
    img  = ft.Image(
        src=photo["presigned_url"],
        fit=ft.ImageFit.COVER,
        visible=loaded,
        on_load=lambda: set_loaded(True),
    )
    return ft.GestureDetector(
        on_tap=lambda: on_click(photo),
        content=ft.Container(
            content=ft.Stack([img, ft.Container(content=ring, alignment=ft.Alignment.CENTER)]),
            width=200,
            height=200,
            clip_behavior=ft.ClipBehavior.HARD_EDGE,
            border_radius=4,
            bgcolor=ft.Colors.SURFACE_VARIANT,
        ),
    )


@ft.component
def PhotoGallery(api_client, on_photo_click):
    state, set_state = ft.use_state(GalleryState())
    page = ft.context.page

    async def load_more():
        if state.loading or state.done:
            return
        set_state(lambda s: GalleryState(
            photos=s.photos, offset=s.offset, loading=True, done=s.done
        ))
        try:
            new_photos = await api_client.get(
                f"/photos/?limit={LIMIT}&offset={state.offset}"
            )
        except Exception:
            set_state(lambda s: GalleryState(
                photos=s.photos, offset=s.offset, loading=False, done=s.done
            ))
            return

        set_state(lambda s: GalleryState(
            photos=s.photos + new_photos,
            offset=s.offset + len(new_photos),
            loading=False,
            done=len(new_photos) < LIMIT,
        ))

    def handle_scroll(e: ft.OnScrollEvent):
        if e.pixels >= e.max_scroll_extent * 0.8:
            page.run_task(load_more)

    ft.use_effect(lambda: (page.run_task(load_more), lambda: None), [])

    tiles = [PhotoTile(photo=p, on_click=on_photo_click) for p in state.photos]
    if state.loading:
        tiles.append(ft.Container(
            content=ft.ProgressRing(),
            alignment=ft.Alignment.CENTER,
            width=200,
            height=200,
        ))

    return ft.GridView(
        controls=tiles,
        runs_count=5,
        max_extent=200,
        spacing=4,
        run_spacing=4,
        expand=True,
        on_scroll=handle_scroll,
        scroll_interval=50,
    )
```

## Optimistic UI Updates

**Optimistic updates** apply a UI change immediately (before the server has confirmed it) and roll back only if the server returns an error. The result is an interface that feels instantaneous: the user sees the effect of their action without waiting for a network round-trip.

Two common operations benefit from this pattern in the photo app:

- **Deleting a photo.** Remove the tile from the gallery immediately. Call `DELETE /photos/{id}` in the background. If the server returns `4xx` or `5xx`, re-insert the tile and show a `SnackBar` error.
- **Naming a person.** Update the displayed label immediately. Call `PATCH /people/{id}` with the new name. If the server fails, revert the label and notify the user.

The pattern in pseudocode:

```python
old_state = capture_state()
apply_optimistic_update()
try:
    await api.action()
except Exception:
    revert_to(old_state)
    show_error("Action failed — please try again")
```

Demonstrating the `httpx` side of an optimistic delete call:

In [ ]:
async def delete_photo_demo(api: APIClient, photo_id: int):
    """
    Simulate an optimistic delete: records the intent, attempts the server call,
    and indicates whether to revert.
    """
    try:
        await api.delete(f"/photos/{photo_id}")
        print(f"Photo {photo_id} deleted successfully — update is permanent")
    except httpx.HTTPStatusError as e:
        print(f"Server error {e.response.status_code} — revert UI and notify user")
    except httpx.ConnectError:
        print("Network unreachable — revert UI and notify user")


token  = create_access_token(sub="demo")
client = APIClient(base_url="http://localhost:8000", token=token)
await delete_photo_demo(client, photo_id=999)

The Flet component implementation of optimistic delete, where the `state` object is a Flet observable that automatically triggers re-renders:

```{.python filename="src/gallery_delete.py"}
import flet as ft


async def handle_delete(photo_id: int, state, api_client):
    """Optimistically remove a photo from state, roll back on server error."""
    page = ft.context.page

    # 1. Capture current state for possible rollback
    original_photos = list(state.photos)

    # 2. Apply optimistic update immediately
    state.photos = [p for p in state.photos if p["id"] != photo_id]

    # 3. Attempt server call
    try:
        await api_client.delete(f"/photos/{photo_id}")
    except Exception:
        # 4. Revert and notify
        state.photos = original_photos
        page.show_snack_bar(
            ft.SnackBar(
                content=ft.Text("Could not delete photo — please try again."),
                bgcolor=ft.Colors.ERROR_CONTAINER,
            )
        )
```

:::{.callout-note}
The `state.photos = [p for p in ...]` assignment (rather than in-place list mutation) is required for Flet observables to detect the change and trigger a re-render. In-place mutation like `state.photos.remove(p)` does work with `facnet_pytorch` observables because Flet wraps list methods, but reassignment is the more explicit and portable pattern.

:::

## Deep Link: Photo Detail View

Tapping a thumbnail navigates to a detail view that shows the full-resolution photo, EXIF metadata, an AI summary, and a "similar photos" horizontal strip. Navigation uses `page.go(f"/photos/{photo_id}")` from the gallery; the back button returns to the gallery, and the gallery preserves scroll position because its state lives in a Flet observable that persists across route transitions.

The detail view fetches three resources in parallel using `asyncio.gather`:
1. `GET /photos/{id}`: full photo metadata including a presigned S3 URL for the high-resolution image.
2. `GET /photos/{id}/exif`: raw EXIF dictionary rendered as a two-column table.
3. `GET /photos/{id}/similar?limit=10`: list of similar photos for the horizontal scroll strip.

```{.python filename="src/detail.py"}
import flet as ft
import asyncio
from dataclasses import dataclass, field


@dataclass
class DetailState:
    photo      : dict | None = None
    exif       : dict | None = None
    similar    : list[dict]  = field(default_factory=list)
    loading    : bool        = True


@ft.component
def ExifTable(exif: dict):
    rows = [
        ft.DataRow(cells=[
            ft.DataCell(ft.Text(k, weight=ft.FontWeight.BOLD, size=12)),
            ft.DataCell(ft.Text(str(v), size=12, selectable=True)),
        ])
        for k, v in exif.items()
    ]
    return ft.DataTable(
        columns=[
            ft.DataColumn(ft.Text("Field")),
            ft.DataColumn(ft.Text("Value")),
        ],
        rows=rows,
        column_spacing=20,
    )


@ft.component
def SimilarStrip(photos: list[dict], on_click):
    tiles = [
        ft.GestureDetector(
            on_tap=lambda p=p: on_click(p),
            content=ft.Image(
                src=p["presigned_url"],
                width=120,
                height=120,
                fit=ft.ImageFit.COVER,
                border_radius=4,
            ),
        )
        for p in photos
    ]
    return ft.Row(
        controls=tiles,
        scroll=ft.ScrollMode.AUTO,
        spacing=6,
    )


@ft.component
def PhotoDetail(photo_id: int, api_client, on_back):
    state, set_state = ft.use_state(DetailState())
    page = ft.context.page

    async def fetch_all():
        try:
            photo, exif, similar = await asyncio.gather(
                api_client.get(f"/photos/{photo_id}"),
                api_client.get(f"/photos/{photo_id}/exif"),
                api_client.get(f"/photos/{photo_id}/similar?limit=10"),
            )
        except Exception:
            set_state(DetailState(loading=False))
            return
        set_state(DetailState(photo=photo, exif=exif, similar=similar, loading=False))

    ft.use_effect(lambda: (page.run_task(fetch_all), lambda: None), [photo_id])

    if state.loading:
        return ft.Container(
            content=ft.ProgressRing(),
            alignment=ft.Alignment.CENTER,
            expand=True,
        )

    if state.photo is None:
        return ft.Text("Photo not found.", color=ft.Colors.ERROR)

    return ft.Column(
        scroll=ft.ScrollMode.AUTO,
        expand=True,
        controls=[
            ft.AppBar(
                leading=ft.IconButton(ft.Icons.ARROW_BACK, on_click=lambda: on_back()),
                title=ft.Text(state.photo.get("taken_at", "Photo"), size=14),
                bgcolor=ft.Colors.SURFACE_CONTAINER_HIGHEST,
            ),
            ft.Image(
                src=state.photo["presigned_url"],
                fit=ft.ImageFit.CONTAIN,
                expand=True,
            ),
            ft.Text("EXIF Metadata", weight=ft.FontWeight.BOLD, size=14),
            ExifTable(exif=state.exif or {}),
            ft.Divider(),
            ft.Text("Similar Photos", weight=ft.FontWeight.BOLD, size=14),
            SimilarStrip(
                photos=state.similar,
                on_click=lambda p: page.go(f"/photos/{p['id']}"),
            ),
        ],
    )
```

**Remark.** The `on_click=lambda p=p: on_click(p)` pattern in `SimilarStrip` is a Python closure gotcha: without the `p=p` default argument, all lambdas in the list comprehension would capture the *last* value of `p` at the time the lambda is called. The default argument binds `p` at definition time.

:::{.callout-tip}
Pass `timeout=httpx.Timeout(connect=3.0, read=30.0)` to `httpx.AsyncClient` for the presigned URL image loads. S3 presigned URL generation is fast but the first-byte latency for large images can be several seconds over a slow connection.

:::

---

■